# Модуль 0. Введение: что такое сериализация и зачем нужен `joblib`

## Зачем этот модуль

Прежде чем учиться сохранять модели машинного обучения или ускорять код, нужно понять одну базовую идею: **объекты в Python живут только пока работает программа**. Как только скрипт завершился — всё, что было в оперативной памяти, исчезает. Чтобы модель, которую вы обучали три часа, не превратилась в ничто после закрытия ноутбука, нужно научиться **сериализовать** — то есть переводить объекты из памяти в файлы, которые можно открыть позже.

Этот модуль объясняет эту идею с нуля и знакомит с библиотекой `joblib`.

## 1. Что такое сериализация

### Простая аналогия

Представьте, что вы нарисовали сложную картину на доске. Доска — это оперативная память компьютера. Картина красивая, но стоит вытереть доску — всё пропало.

Чтобы сохранить картину, вы делаете **фотографию** и кладёте её в альбом. Фотография — это файл на диске. Позже вы можете достать альбом, посмотреть на фотографию и восстановить картину на новой доске.

**Сериализация** — это процесс «фотографирования» объекта из памяти в файл.  
**Десериализация** — это процесс «восстановления картины по фотографии» обратно в память.

### Техническое определение

**Сериализация** (serialization, marshalling, pickling) — преобразование объекта из оперативной памяти в последовательность байтов, которую можно записать на диск, передать по сети или сохранить в базе данных.

**Десериализация** (deserialization, unpickling) — обратный процесс: из последовательности байтов восстанавливается исходный объект в памяти.

### Что можно сериализовать

В Python можно сериализовать почти всё:
- числа, строки, списки, словари
- функции и классы
- объекты `numpy`-массивов
- обученные модели `sklearn`
- целые пайплайны предобработки данных

### Пример «вручную» (без библиотек)

Представьте, что у вас есть словарь:

In [ ]:
model_info = {
    'name': 'RandomForest',
    'accuracy': 0.95,
    'features': ['age', 'income', 'city']
}

Вы можете записать его в JSON:

In [ ]:
import json

with open('model_info.json', 'w') as f:
    json.dump(model_info, f)

А потом прочитать:

In [ ]:
with open('model_info.json', 'r') as f:
    loaded = json.load(f)

Но JSON не умеет сохранять **сложные объекты**: функции, классы, обученные модели sklearn, numpy-массивы. Для этого нужны специальные инструменты.

## 2. Стандартный инструмент Python: модуль `pickle`

Python поставляется с встроенным модулем `pickle`, который умеет сериализовать почти любые Python-объекты.

### Как работает `pickle`

In [ ]:
import pickle

# Есть объект
data = {'numbers': [1, 2, 3], 'name': 'test'}

# Сериализация: объект -> файл
with open('data.pkl', 'wb') as f:  # 'wb' — write binary (байты!)
    pickle.dump(data, f)

# Десериализация: файл -> объект
with open('data.pkl', 'rb') as f:  # 'rb' — read binary
    loaded_data = pickle.load(f)

print(loaded_data)  # {'numbers': [1, 2, 3], 'name': 'test'}

### Ограничения `pickle`

| Проблема | Почему это важно |
|----------|----------------|
| **Медленно на numpy-массивах** | В ML данные — это огромные массивы чисел. `pickle` сохраняет их как обычные Python-списки, что занимает много места и времени |
| **Один файл** | Нельзя сохранить объект так, чтобы части загружались по требованию |
| **Нет встроенного сжатия** | Файлы получаются большими, особенно для моделей с миллионами параметров |
| **Небезопасность** | `pickle.load()` выполняет произвольный код из файла. Загрузка чужого `.pkl` — это как запускать незнакомую программу |

### Пример: сохранение numpy-массива через `pickle`

In [ ]:
import numpy as np
import pickle

arr = np.random.rand(1_000_000)  # миллион чисел

with open('array_pickle.pkl', 'wb') as f:
    pickle.dump(arr, f)

# Размер файла будет ~7.6 МБ
# Время записи и чтения заметное

`pickle` справляется, но не оптимально. Именно поэтому в мире машинного обучения придумали `joblib`.

## 3. Знакомство с `joblib`

### История и назначение

`joblib` — это библиотека, изначально созданная для экосистемы **SciPy** (библиотеки научных вычислений) и **scikit-learn**. Она решает те же задачи, что и `pickle`, но **оптимизирована под ML-рабочий процесс**.

### Три кита `joblib`

Библиотека состоит из трёх независимых, но связанных инструментов:

| Инструмент | Задача | Аналогия |
|------------|--------|----------|
| **`dump` / `load`** | Сохранение и загрузка объектов | Фотоаппарат для объектов |
| **`Memory`** | Кеширование результатов функций | Блокнот, куда записываешь ответы, чтобы не считать заново |
| **`Parallel` + `delayed`** | Распараллеливание циклов | Нанять бригаду рабочих вместо одного |

В этом модуле мы познакомимся только с первым инструментом — `dump`/`load`. Остальные — в следующих модулях.

### Почему `joblib` лучше `pickle` для ML

| Критерий | `pickle` | `joblib` |
|----------|----------|----------|
| Скорость на numpy-массивах | Медленно | Быстро (оптимизировано под numpy) |
| Сжатие | Нет | Встроено (`zlib`, `gzip`, `lzma`) |
| Memory mapping | Нет | Есть (`mmap_mode`) |
| Размер файла | Большой | Меньше при сжатии |
| Сохранение нескольких объектов | В один файл | Можно в структуру с оптимизацией |

## 4. Установка и первая проверка

### Установка

In [ ]:
pip install joblib

Или через `conda`:

In [ ]:
conda install -c conda-forge joblib

### Первая проверка: сохраняем простой объект

In [ ]:
from joblib import dump, load

# Простой словарь
config = {
    'model_type': 'RandomForest',
    'n_estimators': 100,
    'random_state': 42
}

# Сериализация
dump(config, 'config.joblib')

# Десериализация
loaded_config = load('config.joblib')

print(loaded_config)
# {'model_type': 'RandomForest', 'n_estimators': 100, 'random_state': 42}

### Сохранение numpy-массива

In [1]:
import numpy as np
from joblib import dump, load

arr = np.random.rand(1_000_000)

# Сохраняем
dump(arr, 'array.joblib')

# Загружаем
loaded_arr = load('array.joblib')

print(type(loaded_arr))        # <class 'numpy.ndarray'>
print(loaded_arr.shape)        # (1000000,)
print(np.array_equal(arr, loaded_arr))  # True

<class 'numpy.ndarray'>
(1000000,)
True


## 5. Наглядное сравнение: `pickle` vs `joblib`

Давайте проведём честное сравнение на одном и том же массиве.

In [6]:
import numpy as np
import pickle
import os
from joblib import dump
import time

# Создаём большой массив
data = np.random.rand(50_000_000)  # 50 миллионов чисел

# --- pickle ---
start = time.time()
with open('data_pickle.pkl', 'wb') as f:
    pickle.dump(data, f)
time_pickle = time.time() - start
size_pickle = os.path.getsize('data_pickle.pkl')

# --- joblib ---
start = time.time()
dump(data, 'data_joblib.joblib')
time_joblib = time.time() - start
size_joblib = os.path.getsize('data_joblib.joblib')

print(f"pickle:  {time_pickle:.3f} сек, {size_pickle / 1024 / 1024:.2f} МБ")
print(f"joblib:  {time_joblib:.3f} сек, {size_joblib / 1024 / 1024:.2f} МБ")

pickle:  1.183 сек, 381.47 МБ
joblib:  0.328 сек, 381.47 МБ


## 6. Практика: задание

### Задание 0.1: «Первое сохранение»

1. Создайте сложный вложенный словарь, который мог бы описывать конфигурацию ML-эксперимента:
   - название модели
   - список гиперпараметров
   - список фичей (строки)
   - словарь с метриками (`accuracy`, `f1_score`)

2. Сохраните его через `joblib.dump()` в файл `experiment_config.joblib`.

3. Создайте **новый** Python-скрипт (или новую ячейку в Jupyter), который:
   - загружает этот файл через `joblib.load()`
   - выводит на экран содержимое
   - проверяет, что тип загруженного объекта — `dict`

4. Сравните размер файла `experiment_config.joblib` с тем, что получился бы при сохранении через `json`. (Подсказка: `json` сохранит меньше байт для простых типов, но не сможет сохранить, например, функции или numpy-массивы.)

### Эталонное решение 0.1:

In [ ]:
from joblib import dump, load
import os

# 1. Создаём конфиг
config = {
    'model_name': 'GradientBoosting',
    'hyperparams': {
        'n_estimators': 200,
        'learning_rate': 0.05,
        'max_depth': 5
    },
    'features': ['age', 'income', 'city_encoded', 'days_since_last_purchase'],
    'metrics': {
        'accuracy': 0.91,
        'f1_score': 0.88,
        'roc_auc': 0.94
    }
}

# 2. Сохраняем
dump(config, 'experiment_config.joblib')
print(f"Файл создан, размер: {os.path.getsize('experiment_config.joblib')} байт")

# 3. Загружаем в новом месте
loaded = load('experiment_config.joblib')
print(type(loaded))  # <class 'dict'>
print(loaded['model_name'])  # GradientBoosting

### Задание 0.2: «Скорость numpy»

1. Создайте массив `numpy` размером 10_000_000 случайных чисел.
2. Замерьте время сохранения через `pickle` и через `joblib`.
3. Замерьте время загрузки для обоих вариантов.
4. Выведите результат в виде таблицы.

## 7. Вопросы для самопроверки

1. **Что такое сериализация в двух словах?**  
   *(Ответ: перевод объекта из оперативной памяти в формат, который можно сохранить на диск.)*

2. **Почему `pickle` плохо справляется с numpy-массивами?**  
   *(Ответ: он не использует оптимизированные форматы хранения массивов, а записывает их как обычные Python-объекты.)*

3. **Какие три основные задачи решает `joblib`?**  
   *(Ответ: сериализация объектов, кеширование функций, параллельное выполнение циклов.)*

4. **Почему `joblib.load()` теоретически опасен для файлов из интернета?**  
   *(Ответ: потому что при десериализации может выполниться произвольный код, встроенный в файл.)*

5. **В каком режиме открываются файлы для `pickle` и `joblib`?**  
   *(Ответ: в бинарном — `'wb'` для записи, `'rb'` для чтения.)*

## Итоги модуля

- **Сериализация** — это способ сохранить объект из памяти в файл и восстановить его позже.
- В Python есть встроенный `pickle`, но он не оптимален для ML-задач.
- **`joblib`** — это библиотека, заточенная под научные вычисления и машинное обучение. Она быстрее работает с numpy-массивами и умеет сжимать файлы.
- Основные функции: `dump()` — сохранить, `load()` — загрузить.
- В следующем модуле мы научимся сохранять и загружать **настоящие обученные модели sklearn**.